# Tensorflow Object Detection API and AWS Sagemaker

In this notebook, you will train and evaluate different models using the [Tensorflow Object Detection API](https://tensorflow-object-detection-api-tutorial.readthedocs.io/en/latest/) and [AWS Sagemaker](https://aws.amazon.com/sagemaker/). 

If you ever feel stuck, you can refer to this [tutorial](https://aws.amazon.com/blogs/machine-learning/training-and-deploying-models-using-tensorflow-2-with-the-object-detection-api-on-amazon-sagemaker/).

## Dataset

We are using the [Waymo Open Dataset](https://waymo.com/open/) for this project. The dataset has already been exported using the tfrecords format. The files have been created following the format described [here](https://tensorflow-object-detection-api-tutorial.readthedocs.io/en/latest/training.html#create-tensorflow-records). You can find data stored on [AWS S3](https://aws.amazon.com/s3/), AWS Object Storage. The images are saved with a resolution of 640x640.

In [2]:
%%capture
%pip install tensorflow_io sagemaker -U

In [3]:
%pip uninstall -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops
%pip install --no-cache-dir "sagemaker<3"

Found existing installation: sagemaker 3.0
Uninstalling sagemaker-3.0:
  Successfully uninstalled sagemaker-3.0
Found existing installation: sagemaker-core 2.4.0
Uninstalling sagemaker-core-2.4.0:
  Successfully uninstalled sagemaker-core-2.4.0
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 80.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [sagemaker]/2 [sagemaker]
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os 
import sagemaker 
from sagemaker.estimator import Estimator 
from framework import CustomFramework

Unable to load JumpStart region config.
Traceback (most recent call last):
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/jumpstart/constants.py", line 69, in _load_region_config
    with open(filepath) as f:
         ^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/sagemaker/jumpstart/region_config.json'


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [5]:
# role = sagemaker.get_execution_role()
# print(role)
from sagemaker.session import get_execution_role

role = get_execution_role()
print(role)

arn:aws:iam::172148125245:role/service-role/AmazonSageMaker-ExecutionRole-20260127T213109


In [6]:
# The train and val paths below are public S3 buckets created by Udacity for this project
inputs = {'train': 's3://cd2688-object-detection-tf2/train/', 
          'val': 's3://cd2688-object-detection-tf2/val/'} 

# Insert path of a folder in your personal S3 bucket to store tensorboard logs.
tensorboard_s3_prefix = 's3://chapter2-yuri-bucket/logs/'

## Container

To train the model, you will first need to build a [docker](https://www.docker.com/) container with all the dependencies required by the TF Object Detection API. The code below does the following:
* clone the Tensorflow models repository
* get the exporter and training scripts from the repository
* build the docker image and push it 
* print the container name

In [6]:
%%bash

# clone the repo and get the scripts
git clone https://github.com/tensorflow/models.git docker/models

# get model_main and exporter_main files from TF2 Object Detection GitHub repository
cp docker/models/research/object_detection/exporter_main_v2.py source_dir 
cp docker/models/research/object_detection/model_main_tf2.py source_dir

fatal: destination path 'docker/models' already exists and is not an empty directory.


In [ ]:
# build and push the docker image. This code can be commented out after being run once.
# This will take around 10 mins.
image_name = 'tf2-object-detection'
!sh ./docker/build_and_push.sh $image_name

In [7]:
# display the container name
with open (os.path.join('docker', 'ecr_image_fullname.txt'), 'r') as f:
    container = f.readlines()[0][:-1]

print(container)

172148125245.dkr.ecr.us-east-1.amazonaws.com/tf2-object-detection:20260128143237


## Pre-trained model from model zoo

As often, we are not training from scratch and we will be using a pretrained model from the TF Object Detection model zoo. You can find pretrained checkpoints [here](https://github.com/tensorflow/models/blob/master/research/object_detection/g3doc/tf2_detection_zoo.md). Because your time is limited for this project, we recommend to only experiment with the following models:
* SSD MobileNet V2 FPNLite 640x640	
* SSD ResNet50 V1 FPN 640x640 (RetinaNet50)	
* Faster R-CNN ResNet50 V1 640x640	
* EfficientDet D1 640x640	
* Faster R-CNN ResNet152 V1 640x640	

In the code below, the EfficientDet D1 model is downloaded and extracted. This code should be adjusted if you were to experiment with other architectures.

In [8]:
%%bash
mkdir /tmp/checkpoint
mkdir source_dir/checkpoint
wget -O /tmp/efficientdet.tar.gz http://download.tensorflow.org/models/object_detection/tf2/20200711/efficientdet_d1_coco17_tpu-32.tar.gz
tar -zxvf /tmp/efficientdet.tar.gz --strip-components 2 --directory source_dir/checkpoint efficientdet_d1_coco17_tpu-32/checkpoint

mkdir: cannot create directory ‘source_dir/checkpoint’: File exists
--2026-01-28 16:05:00--  http://download.tensorflow.org/models/object_detection/tf2/20200711/efficientdet_d1_coco17_tpu-32.tar.gz
Resolving download.tensorflow.org (download.tensorflow.org)... 192.178.155.207, 142.251.179.207, 64.233.180.207, ...
Connecting to download.tensorflow.org (download.tensorflow.org)|192.178.155.207|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 51839363 (49M) [application/x-tar]
Saving to: ‘/tmp/efficientdet.tar.gz’

     0K .......... .......... .......... .......... ..........  0% 12.9M 4s
    50K .......... .......... .......... .......... ..........  0% 22.3M 3s
   100K .......... .......... .......... .......... ..........  0% 25.6M 3s
   150K .......... .......... .......... .......... ..........  0% 26.0M 2s
   200K .......... .......... .......... .......... ..........  0% 25.0M 2s
   250K .......... .......... .......... .......... ..........  0% 25.3M 2s
  

efficientdet_d1_coco17_tpu-32/checkpoint/ckpt-0.data-00000-of-00001
efficientdet_d1_coco17_tpu-32/checkpoint/checkpoint
efficientdet_d1_coco17_tpu-32/checkpoint/ckpt-0.index


## Edit pipeline.config file

The [`pipeline.config`](source_dir/pipeline.config) in the `source_dir` folder should be updated when you experiment with different models. The different config files are available [here](https://github.com/tensorflow/models/tree/master/research/object_detection/configs/tf2).

>Note: The provided `pipeline.config` file works well with the `EfficientDet` model. You would need to modify it when working with other models.

## Launch Training Job

Now that we have a dataset, a docker image and some pretrained model weights, we can launch the training job. To do so, we create a [Sagemaker Framework](https://sagemaker.readthedocs.io/en/stable/frameworks/index.html), where we indicate the container name, name of the config file, number of training steps etc.

The `run_training.sh` script does the following:
* train the model for `num_train_steps` 
* evaluate over the val dataset
* export the model

Different metrics will be displayed during the evaluation phase, including the mean average precision. These metrics can be used to quantify your model performances and compare over the different iterations.

You can also monitor the training progress by navigating to **Training -> Training Jobs** from the Amazon Sagemaker dashboard in the Web UI.

In [9]:
tensorboard_output_config = sagemaker.debugger.TensorBoardOutputConfig(
    s3_output_path=tensorboard_s3_prefix,
    container_local_output_path='/opt/training/'
)

estimator = CustomFramework(
    role=role,
    image_uri=container,
    entry_point='run_training.sh',
    source_dir='source_dir/',
    hyperparameters={
        "model_dir": "/opt/training",        
        "pipeline_config_path": "pipeline.config",
        "num_train_steps": "2000",    
        "sample_1_of_n_eval_examples": "1"
    },
    instance_count=1,
    instance_type='ml.g5.xlarge',
    tensorboard_output_config=tensorboard_output_config,
    disable_profiler=True,
    base_job_name='tf2-object-detection'
)

estimator.fit(inputs)

INFO:sagemaker:Creating training-job with name: tf2-object-detection-2026-01-28-16-05-02-378


2026-01-28 16:05:05 Starting - Starting the training job
2026-01-28 16:05:05 Pending - Training job waiting for capacity.........
2026-01-28 16:06:10 Pending - Preparing the instances for training...
2026-01-28 16:06:56 Downloading - Downloading the training image............
2026-01-28 16:08:52 Training - Training image download completed. Training in progress.2026-01-28 16:09:01,454 sagemaker-training-toolkit INFO     Provided path: /opt/ml/code  is empty, unzipping
2026-01-28 16:09:02,556 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-01-28 16:09:02,591 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-01-28 16:09:02,627 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-01-28 16:09:02,640 sagemaker-training-toolkit INFO     Invoking user script
Training Env:
{
    "additional_framework_parameters": {},
    "channel_input_dirs": {
        "train": "/o

You should be able to see your model training in the AWS webapp as shown below:
![ECR Example](../data/example_trainings.png)


## Improve on the initial model

Most likely, this initial experiment did not yield optimal results. However, you can make multiple changes to the `pipeline.config` file to improve this model. One obvious change consists in improving the data augmentation strategy. The [`preprocessor.proto`](https://github.com/tensorflow/models/blob/master/research/object_detection/protos/preprocessor.proto) file contains the different data augmentation method available in the Tf Object Detection API. Justify your choices of augmentations in the write-up.

Keep in mind that the following are also available:
* experiment with the optimizer: type of optimizer, learning rate, scheduler etc
* experiment with the architecture. The Tf Object Detection API model zoo offers many architectures. Keep in mind that the pipeline.config file is unique for each architecture and you will have to edit it.
* visualize results on the test frames using the `2_deploy_model` notebook available in this repository.

In the cell below, write down all the different approaches you have experimented with, why you have chosen them and what you would have done if you had more time and resources. Justify your choices using the tensorboard visualizations (take screenshots and insert them in your write-up), the metrics on the evaluation set and the generated animation you have created with [this tool](../2_run_inference/2_deploy_model.ipynb).

In [50]:
%%bash
mkdir -p /tmp/checkpoint
mkdir -p source_dir/checkpoint

wget -O /tmp/ssd_mobilenet.tar.gz \
http://download.tensorflow.org/models/object_detection/tf2/20200711/ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8.tar.gz

tar -zxvf /tmp/ssd_mobilenet.tar.gz \
--strip-components 2 \
--directory source_dir/checkpoint \
ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8/checkpoint

--2026-01-28 20:44:46--  http://download.tensorflow.org/models/object_detection/tf2/20200711/ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8.tar.gz
Resolving download.tensorflow.org (download.tensorflow.org)... 142.251.16.207, 142.251.163.207, 142.251.167.207, ...
Connecting to download.tensorflow.org (download.tensorflow.org)|142.251.16.207|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 20518283 (20M) [application/x-tar]
Saving to: ‘/tmp/ssd_mobilenet.tar.gz’

     0K .......... .......... .......... .......... ..........  0% 19.0M 1s
    50K .......... .......... .......... .......... ..........  0% 26.2M 1s
   100K .......... .......... .......... .......... ..........  0% 11.3M 1s
   150K .......... .......... .......... .......... ..........  0% 40.6M 1s
   200K .......... .......... .......... .......... ..........  1% 41.1M 1s
   250K .......... .......... .......... .......... ..........  1% 60.6M 1s
   300K .......... .......... .......... .......... ..

ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8/checkpoint/ckpt-0.data-00000-of-00001
ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8/checkpoint/checkpoint
ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8/checkpoint/ckpt-0.index


In [52]:
tensorboard_output_config = sagemaker.debugger.TensorBoardOutputConfig(
    s3_output_path=tensorboard_s3_prefix,
    container_local_output_path='/opt/training/'
)

estimator = CustomFramework(
    role=role,
    image_uri=container,
    entry_point='run_training.sh',
    source_dir='source_dir/',
    hyperparameters={
        "model_dir": "/opt/ml/model",        
        "pipeline_config_path": "pipeline.config",
        "num_train_steps": "2000",    
        "sample_1_of_n_eval_examples": "1"
    },
    instance_count=1,
    instance_type='ml.g5.xlarge',
    tensorboard_output_config=tensorboard_output_config,
    disable_profiler=True,
    base_job_name='tf2-object-detection'
)

estimator.fit(inputs)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: tf2-object-detection-2026-01-28-21-10-25-293


2026-01-28 21:10:26 Starting - Starting the training job...
2026-01-28 21:10:53 Pending - Training job waiting for capacity.........
2026-01-28 21:12:32 Downloading - Downloading input data...
2026-01-28 21:12:46 Downloading - Downloading the training image............
2026-01-28 21:14:38 Training - Training image download completed. Training in progress.2026-01-28 21:14:44,709 sagemaker-training-toolkit INFO     Provided path: /opt/ml/code  is empty, unzipping
2026-01-28 21:14:45,471 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-01-28 21:14:45,506 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-01-28 21:14:45,541 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-01-28 21:14:45,554 sagemaker-training-toolkit INFO     Invoking user script
Training Env:
{
    "additional_framework_parameters": {},
    "channel_input_dirs": {
        "train": "/opt/ml/i

In [46]:
%%bash
mkdir -p /tmp/checkpoint
mkdir -p source_dir/checkpoint

wget -O /tmp/faster_rcnn_resnet50.tar.gz \
  http://download.tensorflow.org/models/object_detection/tf2/20200711/faster_rcnn_resnet50_v1_640x640_coco17_tpu-8.tar.gz

tar -zxvf /tmp/faster_rcnn_resnet50.tar.gz \
  --strip-components 2 \
  --directory source_dir/checkpoint \
  faster_rcnn_resnet50_v1_640x640_coco17_tpu-8/checkpoint

--2026-01-28 19:14:52--  http://download.tensorflow.org/models/object_detection/tf2/20200711/faster_rcnn_resnet50_v1_640x640_coco17_tpu-8.tar.gz
Resolving download.tensorflow.org (download.tensorflow.org)... 142.251.167.207, 142.251.111.207, 142.250.31.207, ...
Connecting to download.tensorflow.org (download.tensorflow.org)|142.251.167.207|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 211996178 (202M) [application/x-tar]
Saving to: ‘/tmp/faster_rcnn_resnet50.tar.gz’

     0K .......... .......... .......... .......... ..........  0% 17.4M 12s
    50K .......... .......... .......... .......... ..........  0% 23.2M 10s
   100K .......... .......... .......... .......... ..........  0% 41.2M 8s
   150K .......... .......... .......... .......... ..........  0% 86.9M 7s
   200K .......... .......... .......... .......... ..........  0%  290M 6s
   250K .......... .......... .......... .......... ..........  0% 54.9M 5s
   300K .......... .......... .......... ..

In [49]:
tensorboard_output_config = sagemaker.debugger.TensorBoardOutputConfig(
    s3_output_path=tensorboard_s3_prefix,
    container_local_output_path='/opt/training/'
)

estimator = CustomFramework(
    role=role,
    image_uri=container,
    entry_point='run_training.sh',
    source_dir='source_dir/',
    hyperparameters={
        "model_dir": "/opt/ml/model",        
        "pipeline_config_path": "pipeline_faster_rcnn.config",
        "num_train_steps": "2000",    
        "sample_1_of_n_eval_examples": "1"
    },
    instance_count=1,
    instance_type='ml.g5.xlarge',
    tensorboard_output_config=tensorboard_output_config,
    disable_profiler=True,
    base_job_name='tf2-object-detection'
)

estimator.fit(inputs)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: tf2-object-detection-2026-01-28-20-11-02-999


2026-01-28 20:11:10 Starting - Starting the training job
2026-01-28 20:11:10 Pending - Training job waiting for capacity................................................
2026-01-28 20:19:06 Pending - Preparing the instances for training...
2026-01-28 20:19:35 Downloading - Downloading input data...
2026-01-28 20:19:50 Downloading - Downloading the training image.........
2026-01-28 20:21:36 Training - Training image download completed. Training in progress..2026-01-28 20:21:45,372 sagemaker-training-toolkit INFO     Provided path: /opt/ml/code  is empty, unzipping
2026-01-28 20:21:47,203 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-01-28 20:21:47,240 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-01-28 20:21:47,278 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-01-28 20:21:47,292 sagemaker-training-toolkit INFO     Invoking user script
Training E

### Validation Accuracy (mAP)

| Model                    | Validation mAP |
| ------------------------ | -------------- |
| Efficient Net            | 0.38           |
| SSD MobileNet V2 FPNLite | 0.40           |
| Faster R-CNN ResNet50 V1 | 0.43           |



Among these　three, Faster R-CNN ResNet50 V1 achieved the highest validation mAP (0.43) and is therefore the best-performing model for this task.

### Training loss vs validation loss

Overall, the training loss decreased steadily as training progressed, while the validation loss decreased more slowly and tended to plateau at a higher value than the training loss. This gap indicates some degree of overfitting, which is expected given the limited dataset size and the short training run. The Faster R-CNN model, having higher capacity, is more prone to overfitting; however, in our experiments it still yielded the best validation mAP.

### Was this expected?

Yes. In general, larger-capacity models, such as  Faster R-CNN, often achieve better detection accuracy but can show a larger train–validation loss gap, especially on smaller datasets. Conversely, lighter models, such as SSD MobileNet, usually train faster and generalize more smoothly but may saturate at a lower mAP.

### How to further improve performance

To further improve the performance of the tested models, we could:

1. Increase num_train_steps.

2. Increase data augmentation to reduce overfitting.

3. Tune model-specific hyperparameters, such as anchor sizes/aspect ratios (SSD/FPN), NMS thresholds, and classification loss parameters.

Use higher-resolution inputs or a stronger backbone if resources allow (at the cost of speed/memory).

Improve data quality/quantity (more labeled images, balanced classes, cleaner annotations), which often yields the biggest gains.

Given the results and the accuracy–compute tradeoff, Faster R-CNN ResNet50 is recommended when accuracy is the priority, while SSD MobileNet V2 FPNLite is a good choice when faster training/inference is more important.